In [6]:
schema_sql = """
CREATE TABLE IF NOT EXISTS student_details (
    id          TEXT PRIMARY KEY,
    name        TEXT NOT NULL,
    contact_number INTEGER NOT NULL,
    email_id    TEXT NOT NULL,
    school_id   TEXT NOT NULL,
    grade       TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS cmf_input (
    id           TEXT PRIMARY KEY,
    child_id     TEXT NOT NULL,
    school_id    TEXT NOT NULL,
    submitted_at TEXT NOT NULL,
    status       TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS cmf_metrics (
    id              TEXT PRIMARY KEY,
    input_id        TEXT NOT NULL REFERENCES cmf_input(id),
    processed_date  TEXT NOT NULL,
    wpm             INTEGER NOT NULL,
    wcpm            REAL NOT NULL,
    pronunciation   INTEGER NOT NULL,
    fluency         REAL NOT NULL,
    noise           INTEGER NOT NULL
);

"""

In [7]:
import sqlite3
import json
import os

# Connect to the SQLite database (creates it if it doesn't exist)
DB_PATH = "mydata.db"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.executescript(schema_sql)
print("✅ All three tables created successfully.")
print(f"   Database file: {DB_PATH}")

conn.commit()
conn.close()

✅ All three tables created successfully.
   Database file: mydata.db


In [8]:
import pandas as pd

EXCEL_PATH = "Senior Data Analyst Assignment.xlsx"

# Read the FormResponse sheet
df_form = pd.read_excel(EXCEL_PATH, sheet_name="FormResponse")
print(f"✅ Loaded 'FormResponse' sheet — {len(df_form)} rows, {len(df_form.columns)} columns")
print(f"\nColumns:\n{list(df_form.columns)}")
print(f"\nFirst 5 rows:")
display(df_form.head())

# Store into SQLite
conn = sqlite3.connect(DB_PATH)
df_form.to_sql("form_response", conn, if_exists="replace", index=False)
conn.commit()
conn.close()
print(f"\n✅ Stored in table 'form_response' in {DB_PATH}")

✅ Loaded 'FormResponse' sheet — 37 rows, 8 columns

Columns:
['form_submission_date', 'school_id', 'school_name', 'city', 'state', 'contact_number', 'email_id', 'no_of_students']

First 5 rows:


,form_submission_date,school_id,school_name,city,state,contact_number,email_id,no_of_students
0,2021-03-30 11:09:57,SCH_134213,School Name 134213,Delhi,Delhi,911100222445,test_21459@test.com,1200
1,2021-03-30 11:48:31,SCH_134040,School Name 134040,Amravati,Maharashtra,911100222446,test_21460@test.com,150
2,2021-03-31 03:59:03,SCH_134085,School Name 134085,Greater Noida,Uttar Pradesh,911100222447,test_21461@test.com,162
3,2021-03-31 04:22:44,SCH_134047,School Name 134047,Ghaziabad,Uttar Pradesh,911100222448,test_21462@test.com,150
4,2021-03-31 05:19:28,SCH_134018,School Name 134018,Pune,Maharashtra,911100222449,test_21463@test.com,750



✅ Stored in table 'form_response' in mydata.db


In [9]:
import json

# ── 1. Load JSON data into the SQLite tables ──────────────────────────

conn = sqlite3.connect(DB_PATH)

# Load student_details
with open("StudentDetails.json") as f:
    students = json.load(f)
conn.executemany(
    "INSERT OR IGNORE INTO student_details (id, name, contact_number, email_id, school_id, grade) "
    "VALUES (:id, :name, :contact_number, :email_id, :school_id, :grade)",
    students
)
print(f"✅ Loaded {len(students)} rows into student_details")

# Load cmf_input
with open("CMFInput.json") as f:
    cmf_inputs = json.load(f)
conn.executemany(
    "INSERT OR IGNORE INTO cmf_input (id, child_id, school_id, submitted_at, status) "
    "VALUES (:id, :child_id, :school_id, :submitted_at, :status)",
    cmf_inputs
)
print(f"✅ Loaded {len(cmf_inputs)} rows into cmf_input")

# Load cmf_metrics
with open("CMFMetrics.json") as f:
    cmf_metrics_data = json.load(f)
conn.executemany(
    "INSERT OR IGNORE INTO cmf_metrics (id, input_id, processed_date, wpm, wcpm, pronunciation, fluency, noise) "
    "VALUES (:id, :input_id, :processed_date, :wpm, :wcpm, :pronunciation, :fluency, :noise)",
    cmf_metrics_data
)
print(f"✅ Loaded {len(cmf_metrics_data)} rows into cmf_metrics")

conn.commit()

✅ Loaded 1683 rows into student_details
✅ Loaded 1382 rows into cmf_input
✅ Loaded 1144 rows into cmf_metrics


In [10]:
target_schools = ["SCH_134065", "SCH_134141"]

conn = sqlite3.connect(DB_PATH)

for school_id in target_schools:
    print("═" * 70)
    print(f"           SCHOOL REPORT: {school_id}")
    print("═" * 70)

    # ── School summary row ────────────────────────────────────────
    school_row = pd.read_sql("""
        SELECT
            fr.school_id                                    AS "School ID",
            fr.school_name                                  AS "School Name",
            fr.no_of_students                               AS "Total Student",
            COALESCE(s.cnt, 0)                              AS "Registered Users",
            COALESCE(ci.cnt, 0)                             AS "cmf submitted",
            COALESCE(ci_ok.cnt, 0)                          AS "cmf successful",
            ROUND(COALESCE(cm.avg_wpm, 0), 2)               AS "avg. wpm",
            ROUND(COALESCE(cm.avg_wcpm, 0), 2)              AS "avg. wcpm",
            ROUND(COALESCE(cm.avg_pronunciation, 0), 2)     AS "avg. pronunciation",
            ROUND(COALESCE(cm.avg_fluency, 0), 2)           AS "avg. fluency"
        FROM form_response fr
        LEFT JOIN (SELECT school_id, COUNT(*) AS cnt FROM student_details GROUP BY school_id) s
            ON fr.school_id = s.school_id
        LEFT JOIN (SELECT school_id, COUNT(*) AS cnt FROM cmf_input GROUP BY school_id) ci
            ON fr.school_id = ci.school_id
        LEFT JOIN (SELECT school_id, COUNT(*) AS cnt FROM cmf_input WHERE status = 'Processed' GROUP BY school_id) ci_ok
            ON fr.school_id = ci_ok.school_id
        LEFT JOIN (
            SELECT ci.school_id,
                   AVG(cm.wpm)           AS avg_wpm,
                   AVG(cm.wcpm)          AS avg_wcpm,
                   AVG(cm.pronunciation) AS avg_pronunciation,
                   AVG(cm.fluency)       AS avg_fluency
            FROM cmf_metrics cm
            JOIN cmf_input ci ON cm.input_id = ci.id
            GROUP BY ci.school_id
        ) cm ON fr.school_id = cm.school_id
        WHERE fr.school_id = ?
    """, conn, params=(school_id,))

    print("\n── Schoolwise Report ──────────────────────────────────")
    display(school_row)

    # ── Grade-wise breakdown for this school ────────────────────
    grade_breakdown = pd.read_sql("""
        SELECT
            sd.grade                                        AS "Grade",
            COUNT(DISTINCT sd.id)                           AS "Registered Users",
            COALESCE(ci.cnt, 0)                             AS "cmf submitted",
            COALESCE(ci_ok.cnt, 0)                          AS "cmf successful",
            ROUND(COALESCE(cm.avg_wpm, 0), 2)               AS "avg. wpm",
            ROUND(COALESCE(cm.avg_wcpm, 0), 2)              AS "avg. wcpm",
            ROUND(COALESCE(cm.avg_pronunciation, 0), 2)     AS "avg. pronunciation",
            ROUND(COALESCE(cm.avg_fluency, 0), 2)           AS "avg. fluency"
        FROM student_details sd
        LEFT JOIN (SELECT child_id, COUNT(*) AS cnt FROM cmf_input GROUP BY child_id) ci
            ON sd.id = ci.child_id
        LEFT JOIN (SELECT child_id, COUNT(*) AS cnt FROM cmf_input WHERE status = 'Processed' GROUP BY child_id) ci_ok
            ON sd.id = ci_ok.child_id
        LEFT JOIN (
            SELECT ci.child_id,
                   AVG(cm.wpm)           AS avg_wpm,
                   AVG(cm.wcpm)          AS avg_wcpm,
                   AVG(cm.pronunciation) AS avg_pronunciation,
                   AVG(cm.fluency)       AS avg_fluency
            FROM cmf_metrics cm
            JOIN cmf_input ci ON cm.input_id = ci.id
            GROUP BY ci.child_id
        ) cm ON sd.id = cm.child_id
        WHERE sd.school_id = ?
        GROUP BY sd.grade
        ORDER BY sd.grade
    """, conn, params=(school_id,))

    print("\n── Grade Wise Report ──────────────────────────────────")
    display(grade_breakdown)
    print()

conn.close()


══════════════════════════════════════════════════════════════════════
           SCHOOL REPORT: SCH_134065
══════════════════════════════════════════════════════════════════════

── Schoolwise Report ──────────────────────────────────


,School ID,School Name,Total Student,Registered Users,cmf submitted,cmf successful,avg. wpm,avg. wcpm,avg. pronunciation,avg. fluency
0,SCH_134065,School Name 134065,1200,103,92,64,104.61,86.29,0.76,0.76



── Grade Wise Report ──────────────────────────────────


,Grade,Registered Users,cmf submitted,cmf successful,avg. wpm,avg. wcpm,avg. pronunciation,avg. fluency
0,Grade 1,21,1,1,32.7,8.9,0.19,0.00
1,Grade 2,26,1,0,0.0,0.0,0.00,0.00
2,Grade 3,23,1,1,82.5,57.6,0.83,0.82
3,Grade 4,33,2,1,139.3,129.7,0.91,0.88



══════════════════════════════════════════════════════════════════════
           SCHOOL REPORT: SCH_134141
══════════════════════════════════════════════════════════════════════

── Schoolwise Report ──────────────────────────────────


,School ID,School Name,Total Student,Registered Users,cmf submitted,cmf successful,avg. wpm,avg. wcpm,avg. pronunciation,avg. fluency
0,SCH_134141,School Name 134141,1500,111,124,102,95.74,71.71,0.66,0.78



── Grade Wise Report ──────────────────────────────────


,Grade,Registered Users,cmf submitted,cmf successful,avg. wpm,avg. wcpm,avg. pronunciation,avg. fluency
0,Grade 1,27,6,5,60.26,32.94,0.51,0.67
1,Grade 2,25,1,1,83.20,61.70,0.85,0.81
2,Grade 3,31,2,2,92.65,41.70,0.34,0.80
3,Grade 4,28,2,2,139.30,129.70,0.91,0.88
